# 15 — AgentCore Memory, identidade e ACL atual

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 1 conta AWS

Esta aula não exige fronteiras entre contas. Seus stacks, integrações e permissões precisam estar implantados na conta da plataforma. Ela não funciona automaticamente em uma conta AWS vazia.

**Runner CodeBuild do projeto:** a seleção é declarada no deploy. A jornada `single-account` inclui 00–09 e 15–16; Registry, Terraform e RAG são exigidos somente pelas aulas que os utilizam. Ampliar a seleção exige atualizar a implantação pelo IaC. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 15](../docs/diagramas/svg/15_ingestao_acl_memoria.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "15")
print({"lab": "15", "dependencies": list(dependencies), "region": config["region"]})

### Conectar ao ambiente implantado

Lemos os outputs exatos do CloudFormation. O cliente não cria infraestrutura e não substitui um serviço indisponível.

In [ ]:
lab = AWSWorkshop(CONFIG, domain="platform")
print(lab.preflight())

### Autenticar a pessoa no Cognito

O login usa USER_PASSWORD_AUTH para usuários sintéticos do workshop e SECRET_HASH para o cliente confidencial. Em uma aplicação de usuários finais, use authorization code com PKCE. O código abaixo mostra a chamada Cognito e a verificação de assinatura, emissor, audiência e token_use. Só imprimimos claims selecionadas.

In [ ]:
from santander_demo.aws_workshop.identity import login, verify
display(Code(inspect.getsource(login), language="python"))
display(Code(inspect.getsource(verify), language="python"))
token = lab.login("c", "supervisor")
claims = lab.claims(token)
print({key: claims[key] for key in ["iss", "sub", "aud", "token_use"]})

### Ler a fase implantada pelo IaC

A aula tem cinco transições: inclusão, atualização, revogação, exclusão e reingestão. O pipeline workshop_lifecycle.py atualiza os stacks e inicia esta aula em CodeBuild. O runner só observa a infraestrutura e invoca os serviços; cada fase precisa passar antes de aplicar a próxima, preservando o mesmo session_id.

In [ ]:
import os
from santander_demo.aws_workshop.lifecycle import revision, stage
from santander_demo.rag.deployment import outputs as rag_outputs
rag_config = json.loads((ROOT / config["integrations"]["rag"]["config"]).read_text())
rag_data = rag_outputs(lab.aws, rag_config, "data")
lifecycle_stage = stage(os.environ.get("WORKSHOP_LIFECYCLE_STAGE", "baseline"))
session_id = os.environ.get("WORKSHOP_MEMORY_SESSION", "aula15-" + uuid.uuid4().hex)
assert lab.backend["LifecycleStage"] == lifecycle_stage
assert rag_data["LifecycleStage"] == lifecycle_stage
display(Code((ROOT / "scripts/workshop_lifecycle.py").read_text(), language="python"))
print({"stage": lifecycle_stage, "session_id": session_id,
       "ingestion_job": rag_data["IngestionJobId"]})

### Inspecionar Memory e a implementação da fonte

AgentCore Memory mantém eventos separados por actor/session. actor é derivado de issuer/sub verificados; session_id sozinho não identifica uma pessoa. Guardamos referências para evitar que uma cópia antiga devolva conteúdo após revogação.

In [ ]:
control = lab.aws.client("bedrock-agentcore-control")
memory_resource = control.get_memory(memoryId=lab.backend["MemoryId"])["memory"]
assert memory_resource["status"] == "ACTIVE"
print({"memory_id": memory_resource["id"], "status": memory_resource["status"]})
from santander_demo.aws_workshop.backend import Source
display(Code(inspect.getsource(Source.remember), language="python"))
display(Code(inspect.getsource(Source.recall), language="python"))

### Gravar uma referência no Runtime na primeira fase

O Runtime verifica a ACL de k2 e usa CreateEvent no Memory. As fases seguintes reutilizam esse evento; gravar outra referência após a revogação esconderia a prova de reautorização.

In [ ]:
if lifecycle_stage == "baseline":
    event = await lab.call(token, "especialista___memorizar_referencia",
                          {"tabela": "k2", "session_id": session_id})
    assert event["result"]["stored"] == "reference_only"
    print(event)
else:
    assert os.environ.get("WORKSHOP_MEMORY_SESSION"), "Reutilize a sessão da fase baseline"

### Reautorizar a referência e isolar outra pessoa

C deve receber a versão atual, nunca a cópia antiga. Nas fases revoked e deleted, a referência antiga precisa produzir uma lista vazia. A usa o mesmo session_id e permanece isolado em outro actor.

In [ ]:
own = await lab.call(token, "especialista___recuperar_memoria", {"session_id": session_id})
other = await lab.call(lab.login("a"), "especialista___recuperar_memoria",
                       {"session_id": session_id})
documents = own["result"]["documents"]
assert own["result"]["references_seen"] == 1, "A referência original deve continuar no Memory"
if lifecycle_stage in {"revoked", "deleted"}:
    assert documents == []
else:
    assert [x["table"] for x in documents] == ["k2"]
    assert documents[0]["rows"][0]["revision"] == revision(lifecycle_stage)
assert other["result"]["documents"] == []
assert other["result"]["references_seen"] == 0
print({"own": own, "other": other})

### Conferir ingestão e consultar o documento pela ACL nativa

O provider publica conteúdo e ACL em S3 e aguarda o job de ingestão. F1 muda de versão, perde o grant de C, é excluído e depois reingestado. A consulta atravessa o Gateway da Knowledge Base com JWT Cognito; não usa Retrieve direto com credenciais administrativas.

In [ ]:
import asyncio
from santander_demo.rag.client import _retrieve, checked_results, retrieval_arguments, verify_token
from santander_demo.rag.cognito import login as rag_login
from santander_demo.rag.transport import correlate
from santander_demo.journey.evidence import payload
import time
rag_access = rag_outputs(lab.aws, rag_config, "access")
job = lab.aws.client("bedrock-agent").get_ingestion_job(
    knowledgeBaseId=rag_data["KnowledgeBaseId"], dataSourceId=rag_data["DataSourceId"],
    ingestionJobId=rag_data["IngestionJobId"],
)["ingestionJob"]
assert job["status"] == "COMPLETE"
assert job.get("statistics", {}).get("numberOfDocumentsFailed", 0) == 0
rag_token = rag_login(lab.aws, rag_config, rag_outputs(lab.aws, rag_config, "identity"), "c")
identity = verify_token(rag_token, rag_config, destination="gateway")
request_ids = []
since_ms = int(time.time() * 1000)
for attempt in range(12):
    result = await _retrieve(
        rag_access["GatewayUrl"], rag_token,
        retrieval_arguments(identity, "Documento F1 dispositivo desconhecido CICLO-F1"),
        request_ids,
    )
    assert not result.isError
    chunks = checked_results(payload(result), "c", rag_data["SourceBucket"])
    selected = [chunk for chunk in chunks if chunk["id"] == "F1"]
    if lifecycle_stage == "revoked":
        matched = not any(chunk["domain"] == "fraude" for chunk in chunks)
    elif lifecycle_stage == "deleted":
        matched = not selected
    else:
        marker = f"CICLO-F1-V{revision(lifecycle_stage)}"
        matched = bool(selected) and all(marker in chunk["text"] for chunk in selected)
    if matched:
        break
    await asyncio.sleep(5)
assert matched, "A ingestão terminou, mas a mudança ainda não foi comprovada na consulta"
row = {"case": "c", "gateway_request_ids": request_ids}
for attempt in range(12):
    correlate(lab.aws, rag_access, [row], since_ms)
    if row["gateway_log_correlation_verified"]:
        break
    await asyncio.sleep(5)
assert row["gateway_log_correlation_verified"]
print({"stage": lifecycle_stage, "ingestion_job": rag_data["IngestionJobId"],
       "document_ids": sorted({chunk["id"] for chunk in chunks}), "gateway": row})

Execute `workshop_lifecycle.py` na ordem baseline → updated → revoked → deleted → reingested, com o mesmo `--memory-session`. Cada fase produz um build e um notebook executado. Depois, aplique baseline para restaurar o corpus compartilhado antes da jornada completa. Versões S3 são preservadas; a exclusão didática só alcança F1.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.